# Born Displaced — Python Visualizations
**VIZ 5** Pictogram (explanatory) · **VIZ 6** Trajectory chart (creative) · **VIZ 7** White hat · **VIZ 8** Black hat

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

os.makedirs('plots', exist_ok=True)
df = pd.read_csv('data/master_dataset.csv')
print(df.shape)

(3159, 17)


## VIZ 5 — Pictogram: child mortality in high vs low displacement countries
**Audience**: non-esperti. Ogni punto = 1 bambino su 1.000. Rosso = non sopravvissuto.

In [2]:
high_rate = df[df['total_displaced'] >= 1_000_000]['mortality_1t4'].mean()
low_rate  = df[(df['total_displaced'] <= 10_000) & df['total_displaced'].notna()]['mortality_1t4'].mean()

print(f"High displacement: {high_rate:.1f} deaths per 1,000")
print(f"Low displacement:  {low_rate:.1f} deaths per 1,000")

def dot_grid(rate, n=1000, cols=40):
    n_deaths = int(round(rate))
    xs = [i % cols for i in range(n)]
    ys = [-(i // cols) for i in range(n)]
    colors = ['#C0392B' if i < n_deaths else '#D5DBDB' for i in range(n)]
    labels = ['did not survive' if i < n_deaths else 'survived' for i in range(n)]
    return xs, ys, colors, labels

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f"<b>Paesi ad alto sfollamento</b><br>{high_rate:.1f} morti ogni 1.000 bambini",
        f"<b>Paesi a basso sfollamento</b><br>{low_rate:.1f} morti ogni 1.000 bambini"
    ]
)

for col_i, rate in enumerate([high_rate, low_rate], 1):
    xs, ys, colors, labels = dot_grid(rate)
    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode='markers',
        marker=dict(color=colors, size=8, symbol='circle', line=dict(width=0)),
        hovertext=labels,
        hoverinfo='text',
        showlegend=False
    ), row=1, col=col_i)

fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_layout(
    title=dict(
        text="Ogni punto è un bambino (1–4 anni) su 1.000 · <span style='color:#C0392B'>●</span> non sopravvissuto · <span style='color:#D5DBDB'>●</span> sopravvissuto",
        x=0.5, font=dict(size=14)
    ),
    plot_bgcolor='white', paper_bgcolor='white',
    height=500, width=960
)

fig.write_html('plots/viz5_explanatory_pictogram.html')
fig.show()

High displacement: 26.9 deaths per 1,000
Low displacement:  4.9 deaths per 1,000


## VIZ 6 — Creative: Trajectory chart
Ogni paese è un percorso nello spazio (sfollamento × mortalità) dal 2010 al 2022.  
Il cerchio piccolo = 2010, il cerchio grande = 2022. Si legge come una traiettoria di crisi o di miglioramento.

In [3]:
TOP_N = 15
top_countries = (
    df.groupby('country_name')['total_displaced']
    .mean().dropna().nlargest(TOP_N).index.tolist()
)

traj = (
    df[df['country_name'].isin(top_countries) &
       df['total_displaced'].notna() &
       df['mortality_1t4'].notna()]
    .copy()
)
traj['log_displaced'] = np.log10(traj['total_displaced'].clip(1))
traj = traj.sort_values(['country_name', 'year'])

palette = px.colors.qualitative.Bold + px.colors.qualitative.Safe

fig = go.Figure()

for i, country in enumerate(top_countries):
    cdf = traj[traj['country_name'] == country]
    if len(cdf) < 2:
        continue
    color = palette[i % len(palette)]

    fig.add_trace(go.Scatter(
        x=cdf['log_displaced'],
        y=cdf['mortality_1t4'],
        mode='lines',
        line=dict(color=color, width=1.8),
        showlegend=False,
        hoverinfo='skip'
    ))

    # tutti i punti (piccoli)
    fig.add_trace(go.Scatter(
        x=cdf['log_displaced'],
        y=cdf['mortality_1t4'],
        mode='markers',
        marker=dict(color=color, size=5, opacity=0.5),
        showlegend=False,
        customdata=cdf['year'].values,
        hovertemplate=f'<b>{country}</b><br>Anno: %{{customdata}}<br>log(sfollati): %{{x:.2f}}<br>Mortalità: %{{y:.2f}}<extra></extra>'
    ))

    # punto 2022 (grande, con etichetta paese)
    last = cdf.iloc[-1]
    fig.add_trace(go.Scatter(
        x=[last['log_displaced']],
        y=[last['mortality_1t4']],
        mode='markers+text',
        marker=dict(color=color, size=11, symbol='circle'),
        text=[country.split()[0]],
        textposition='top right',
        textfont=dict(size=9, color=color),
        showlegend=True,
        name=country,
        hovertemplate=f'<b>{country}</b> (2022)<br>log(sfollati): %{{x:.2f}}<br>Mortalità: %{{y:.2f}}<extra></extra>'
    ))

    # punto 2010 (marcatore triangolo)
    first = cdf.iloc[0]
    fig.add_trace(go.Scatter(
        x=[first['log_displaced']],
        y=[first['mortality_1t4']],
        mode='markers',
        marker=dict(color=color, size=8, symbol='triangle-up', opacity=0.7),
        showlegend=False,
        hovertemplate=f'<b>{country}</b> (2010)<extra></extra>'
    ))

fig.update_layout(
    title=dict(
        text="Traiettoria di crisi: sfollamento vs mortalità infantile (2010–2022)<br>"
             "<sup>▲ = 2010 · ● = 2022 · Ogni linea è il percorso di un paese nel tempo</sup>",
        x=0.5, font=dict(size=14)
    ),
    xaxis=dict(title='log₁₀(Sfollati totali)', showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(title='Tasso di mortalità infantile 1–4 anni (per 1.000)', showgrid=True, gridcolor='#eeeeee'),
    plot_bgcolor='white', paper_bgcolor='white',
    height=620, width=1000,
    legend=dict(title='Paese (● = 2022)', font=dict(size=9))
)

fig.write_html('plots/viz6_creative_trajectory.html')
fig.show()

## VIZ 7 — White Hat
Scatter onesto: intervalli di confidenza UNICEF visibili, colore per gruppo di reddito, disclaimer su correlazione ≠ causalità.

In [4]:
wh = df[
    df['total_displaced'].notna() &
    df['mortality_1t4'].notna() &
    df['mortality_1t4_low'].notna() &
    (df['total_displaced'] > 0)
].copy()
wh['log_displaced'] = np.log10(wh['total_displaced'])

wh['income_group'] = pd.qcut(
    wh['gdp_per_capita'].fillna(wh['gdp_per_capita'].median()),
    q=4, labels=['Low income', 'Lower-middle', 'Upper-middle', 'High income']
)

COLORS = {
    'Low income':     '#E74C3C',
    'Lower-middle':   '#E67E22',
    'Upper-middle':   '#3498DB',
    'High income':    '#27AE60'
}

fig = go.Figure()

for group, color in COLORS.items():
    g = wh[wh['income_group'] == group]
    fig.add_trace(go.Scatter(
        x=g['log_displaced'],
        y=g['mortality_1t4'],
        mode='markers',
        name=group,
        marker=dict(color=color, size=5, opacity=0.45),
        error_y=dict(
            type='data', symmetric=False,
            array=(g['mortality_1t4_high'] - g['mortality_1t4']).tolist(),
            arrayminus=(g['mortality_1t4'] - g['mortality_1t4_low']).tolist(),
            thickness=0.7, width=0, visible=True
        ),
        customdata=g[['country_name', 'year']].values,
        hovertemplate='<b>%{customdata[0]}</b> (%{customdata[1]})<br>Sfollati: 10^%{x:.2f}<br>Mortalità: %{y:.2f} per 1.000<extra></extra>'
    ))

m, b = np.polyfit(wh['log_displaced'], wh['mortality_1t4'], 1)
xr = np.linspace(wh['log_displaced'].min(), wh['log_displaced'].max(), 100)
fig.add_trace(go.Scatter(
    x=xr, y=m * xr + b,
    mode='lines', name='Trend lineare (OLS)',
    line=dict(color='#2C3E50', width=1.5, dash='dash')
))

fig.update_layout(
    title=dict(
        text="Associazione tra sfollamento forzato e mortalità infantile (1–4 anni), 2010–2022<br>"
             "<sup>Ogni punto = un paese-anno · Barre di errore = IC 90% UNICEF · Colore = gruppo di reddito (quartile PIL pro capite)</sup>",
        x=0.5, font=dict(size=13)
    ),
    xaxis=dict(title='log₁₀(Sfollati totali)', range=[0, wh['log_displaced'].max() + 0.3],
               showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(title='Mortalità infantile (decessi per 1.000 bambini 1–4 anni)',
               range=[0, wh['mortality_1t4'].max() * 1.1],
               showgrid=True, gridcolor='#eeeeee'),
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(title='Gruppo di reddito'),
    annotations=[dict(
        text="⚠ Correlazione ≠ causalità. I paesi ad alto sfollamento presentano spesso<br>"
             "vulnerabilità strutturali preesistenti non catturate da questo grafico.<br>"
             "Fonte: UNICEF (UN IGME), UNHCR, World Bank · Dati 2010–2022",
        xref='paper', yref='paper', x=0.01, y=0.98,
        showarrow=False, align='left',
        font=dict(size=9, color='#7F8C8D'),
        bgcolor='rgba(255,255,255,0.85)'
    )],
    height=580, width=920
)

fig.write_html('plots/viz7_white_hat.html')
fig.show()

## VIZ 8 — Black Hat
**Manipolazioni applicate:**
1. Asse Y troncato (parte da 3, non da 0) → differenze esagerate visivamente
2. Solo anni 2015–2022, solo paesi con >100k sfollati → cherry-pick
3. Fit polinomiale (grado 2) → trend visivamente più ripido del lineare
4. Nessun intervallo di confidenza
5. Tutti i marker in rosso → risposta emotiva
6. Titolo afferma causalità diretta
7. Annotazione suggerisce una stima causale precisa (inventata)

In [5]:
bh = wh[
    (wh['year'] >= 2015) &
    (wh['total_displaced'] >= 100_000)
].copy()

coeffs = np.polyfit(bh['log_displaced'], bh['mortality_1t4'], 2)
xr_bh = np.linspace(bh['log_displaced'].min(), bh['log_displaced'].max(), 100)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bh['log_displaced'],
    y=bh['mortality_1t4'],
    mode='markers',
    marker=dict(color='#C0392B', size=6, opacity=0.7),
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=xr_bh,
    y=np.polyval(coeffs, xr_bh),
    mode='lines',
    line=dict(color='#922B21', width=3),
    showlegend=False
))

fig.update_layout(
    title=dict(
        text="<b>Lo sfollamento forzato uccide i bambini: una crisi innegabile</b><br>"
             "<sup>Più persone sfollate = più bambini morti. I dati parlano chiaro.</sup>",
        x=0.5, font=dict(size=15, color='#922B21')
    ),
    xaxis=dict(
        title='Sfollamento (scala logaritmica)',
        showgrid=False, zeroline=False
    ),
    yaxis=dict(
        title='Morti per 1.000 bambini',
        range=[3, bh['mortality_1t4'].max() * 1.05],  # ← ASSE TRONCATO
        showgrid=False, zeroline=False
    ),
    annotations=[
        dict(
            text="★ Ogni 100.000 sfollati aggiuntivi<br>causano ~2.3 morti in più per 1.000 bambini",
            xref='paper', yref='paper', x=0.97, y=0.95,
            showarrow=False, align='right',
            font=dict(size=11, color='#922B21'),
            bgcolor='rgba(255,235,235,0.95)',
            bordercolor='#922B21', borderwidth=1
        ),
        dict(
            text="Dati: 2015–2022",
            xref='paper', yref='paper', x=0.01, y=0.01,
            showarrow=False, font=dict(size=8, color='#CCCCCC')
        )
    ],
    plot_bgcolor='#FEF9F9', paper_bgcolor='white',
    height=520, width=820
)

fig.write_html('plots/viz8_black_hat.html')
fig.show()

## Output

In [6]:
import os
for f in sorted(os.listdir('plots')):
    size = os.path.getsize(f'plots/{f}') // 1024
    print(f'  plots/{f}  ({size} KB)')

  plots/viz5_explanatory_pictogram.html  (4795 KB)
  plots/viz6_creative_trajectory.html  (4770 KB)
  plots/viz7_white_hat.html  (4942 KB)
  plots/viz8_black_hat.html  (4750 KB)
